In [2]:
import os
import sys
from pathlib import Path

ROOT = Path(os.path.abspath('')).resolve().parents[2]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import nvitk.core as core
# core.setup(globals())

import nvitk as nv

In [3]:
tof = nv.imread('/home/imarcoss/NetVolumes/Tierra/LAB_VF-ICH/LAB/MCC LAB/_AlbertoFernandez/Proyectos/Aurora_PESA/NIFTI/TOF_3D_FLASH_flc_MR_40001.nii.gz')
print(tof)

Image(shape=(256, 96, 256), dtype=int16, backend=cupy, axes='XYZ', orientation='LAS', name='TOF_3D_FLASH_flc_MR_40001.nii', modality='MR', submodality='TOF_3D_FLASH_flc', rescale_type='DV')


In [6]:
tof = nv.imread('/home/imarcoss/NetVolumes/Tierra/LAB_VF-ICH/LAB/MCC LAB/_AlbertoFernandez/Proyectos/Aurora_PESA/NIFTI/mouse_T2_test.nii.gz')
print(tof)
print(tof.affine)
print(tof.spacing)

Image(shape=(128, 35, 96), dtype=float32, backend=cupy, axes='XYZ', orientation='LAS', name='mouse_T2_test.nii', modality=None, submodality=None, rescale_type='DV')
[[-0.15000001 -0.          0.         10.60000038]
 [-0.          0.5        -0.         -9.55000114]
 [ 0.          0.          0.15000001 -8.5       ]
 [ 0.          0.          0.          1.        ]]
(0.15000000596046448, 0.5, 0.15000000596046448)


In [5]:
print(tof.affine)

[[ 7.80729726e-02  2.81461887e-03 -9.05564521e-04 -8.01018906e+00]
 [-2.82843760e-03  7.49258399e-02 -4.38923277e-02  1.08090544e+00]
 [-3.56415956e-04  2.19479408e-02  1.49955675e-01 -7.04215860e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [ ]:
pt = nv.imread('/home/imarcoss/NetVolumes/Tierra/GENERAL_SC/PESA-Fat/Visit-5-DIXON_PET-CT/IMAGES/202602_Week1/PESA11471769/PET_PESA11471769', force_type='dicom')
pt_rev = nv.imread('/home/imarcoss/NetVolumes/Tierra/GENERAL_SC/PESA-Fat/Visit-5-DIXON_PET-CT/IMAGES/202602_Week1/PESA11471769/PET_PESA11471769', force_type='dicom', revert_scaling=True)


17:49:41 | INFO     | Reverting scanner scaling to obtain raw pixel values (revert_scaling takes priority over rescale_type)
17:49:41 | INFO     | Scaling factors vary across slices. Applying per-slice rescaling to 490 slices
17:49:41 | INFO     | Reverted scanner scaling on 490/490 slices to obtain raw pixel values


In [3]:
nv.imread('/data3/BIOIT_IMAGE/PESA_Fat/DATA/Visit-5-DIXON_PET-CT/RESULTS/202602_Week2/res_post_processing_ct/PESA3755844/CT/ORGANS.nii.gz')

Image(shape=(512, 512, 540), dtype=uint8, backend=cupy, axes='XYZ', orientation='RAS', name='ORGANS.nii', modality='CT', submodality=None, rescale_type='DV')

In [1]:
from pathlib import Path

import pandas as pd

from nvitk.db.xnat_config import load_xnat_profile, resolve_xnat_connection
from nvitk.pipes.qvtpy.stage0_download import (
    DEFAULT_SEQUENCES,
    resolve_subjects_for_xnat_pipeline,
)


def save_pesabrain_qvtpy_subjects_csv(
    output_csv: Path,
    *,
    xnat_config_path: Path = Path("~/nvitk/.nvitk/xnat.json"),
    database_root: Path = Path("~/nvitk/dataset/nvitk-dataset"),
    cohort: str = "PESA-Brain",
    sequences: tuple[str, ...] = DEFAULT_SEQUENCES,
) -> list[str]:
    """Ordered PESA-Brain subjects with all 4 qvtpy sequences → CSV."""
    profile = load_xnat_profile(xnat_config_path)
    conn = resolve_xnat_connection(profile)

    subjects, _conn = resolve_subjects_for_xnat_pipeline(
        subjects=cohort,
        subjects_file=None,
        xnat_config=conn,
        database_root=database_root,
    )

    # resolve_subjects_for_xnat_pipeline already filters via DB when database_root is set
    subjects = sorted(subjects)

    df = pd.DataFrame({"subject_id": subjects})
    output_csv = Path(output_csv).expanduser().resolve()
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)

    print(f"Wrote {len(subjects)} subject(s) → {output_csv}")
    print(f"Required sequences: {', '.join(sequences)}")
    return subjects


save_pesabrain_qvtpy_subjects_csv("/home/imarcoss/DATA/LabVF/PESA-Brain/pesabrain_qvtpy_eligible.csv")

[WARNING] Verify is disabled, this will NOT verify the certificate of SSL connections!
[WARNING] Warnings about invalid certificates will be HIDDEN to avoid spam, but this
[WARNING] means that your connection can be potentially unsafe!
11:20:57 | INFO     | XNAT cohort alias 'PESA-Brain' -> project 'PESA_Brain' (980 subject(s))
/home/imarcoss/nvitk/src/nvitk/db/xnat.py:949: UserWarning: Error querying table assets with SQLite index. Falling back to parquet read.
  assets = repo._load_table_frame("assets", use_sqlite=True)
11:20:57 | INFO     | DB scan pre-filter (PESA_Brain): 527/980 subject(s) have 4dflow_ap, 4dflow_fh, 4dflow_rl, tof
11:20:57 | INFO     |   excluded: PESA10055241, PESA10118761, PESA10169721, PESA1018081, PESA1024144, PESA10252804, PESA1030225, PESA10368400 ... (+445 more)


Wrote 527 subject(s) → /home/imarcoss/DATA/LabVF/PESA-Brain/pesabrain_qvtpy_eligible.csv
Required sequences: TOF, 4DFLOW_AP, 4DFLOW_RL, 4DFLOW_FH


['PESA10017225',
 'PESA100489',
 'PESA1006009',
 'PESA10061584',
 'PESA10067929',
 'PESA1008016',
 'PESA10086976',
 'PESA10112400',
 'PESA10137856',
 'PESA10144225',
 'PESA10195249',
 'PESA10201636',
 'PESA10220809',
 'PESA10227204',
 'PESA10233601',
 'PESA10240000',
 'PESA10272025',
 'PESA10310521',
 'PESA10374841',
 'PESA10404',
 'PESA10407076',
 'PESA10413529',
 'PESA10465225',
 'PESA10497600',
 'PESA10510564',
 'PESA10523536',
 'PESA10569001',
 'PESA10627600',
 'PESA10758400',
 'PESA10778089',
 'PESA10843849',
 'PESA1085764',
 'PESA10903204',
 'PESA10909809',
 'PESA10923025',
 'PESA10936249',
 'PESA10989225',
 'PESA1102500',
 'PESA11048976',
 'PESA11062276',
 'PESA11082241',
 'PESA1108809',
 'PESA11168964',
 'PESA11175649',
 'PESA11215801',
 'PESA11222500',
 'PESA1123600',
 'PESA11276164',
 'PESA11410884',
 'PESA11437924',
 'PESA11471769',
 'PESA11492100',
 'PESA11505664',
 'PESA11546404',
 'PESA11573604',
 'PESA11580409',
 'PESA11600836',
 'PESA11628100',
 'PESA11641744',
 'PESA11